In [0]:

# Read the silver tables, calculate predictors, and build 30-minute stockout targets.
# Output keys are station_id and fetched_at. Predictors and targets are saved separately.
# Feature calculations are imported from station_features.py. This Free Edition version reads the silver tables
# downloaded from the S3 export into workspace.default, and writes its gold tables there.
# This is intended to build the dataset we will use to train and validate our stockout and dock full prediction
# models


# The feature functions are imported from station_features.py, in the same folder as this notebook.

from pyspark.sql import functions as F, Window
from station_features import build_predictors, build_targets


spark.conf.set("spark.sql.session.timeZone", "UTC")
# Scheduled jobs set run_mode=production.
# Manual development runs default to calculating without writing tables.
# We aren't using the pipeline_config code from the main site yet

dbutils.widgets.dropdown(
    "run_mode", "development", ["development", "production"]
)

RUN_MODE = dbutils.widgets.get("run_mode")
PUBLISH = RUN_MODE == "production"

STATUS_TABLE = "workspace.default.silver_station_status"
INFO_TABLE = "workspace.default.silver_station_info"

PREDICTOR_TABLE = "workspace.default.gold_station_predictors"
TARGET_TABLE = "workspace.default.gold_station_targets"

print(f"Run mode: {RUN_MODE}")
print(f"Write gold tables: {PUBLISH}")

LOCAL_TZ = "America/New_York"
HORIZON_MIN = 30
MAX_OBSERVATION_GAP_MIN = 10
MIN_HISTORY_MIN = 20
MICROSECONDS_PER_MINUTE = 60000000

status = spark.table(STATUS_TABLE).select(
    "station_id", "fetched_at", "station_reported_at",
    "num_bikes_available", "num_bikes_disabled",
    "num_docks_available", "num_docks_disabled",
    "is_installed", "is_renting", "is_returning",
)
info = spark.table(INFO_TABLE).select(
    "station_id", "fetched_at", "name", "capacity", "lat", "lon"
).distinct()

# Check missing and duplicate keys in one aggregation per table.
for name, frame in [("status", status), ("information", info)]:
    invalid_keys = frame.groupBy("station_id", "fetched_at").count().filter(
        "station_id IS NULL OR fetched_at IS NULL OR count > 1"
    )
    if invalid_keys.take(1):
        display(invalid_keys.limit(10))
        raise ValueError(f"{name}: missing or duplicate station/poll keys")


# We build the predictors table

predictors = build_predictors(
    status,
    info,
    local_timezone=LOCAL_TZ,
    max_gap_minutes=MAX_OBSERVATION_GAP_MIN,
    min_history_minutes=MIN_HISTORY_MIN,
)
# Write and reload predictors before calculating targets. We do this for speed because
# we don't want to trigger the lazy eval twice

if PUBLISH:
    predictors.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(PREDICTOR_TABLE)
    predictors = spark.table(PREDICTOR_TABLE)

targets = build_targets(
    predictors,
    horizon_minutes=HORIZON_MIN,
    max_gap_minutes=MAX_OBSERVATION_GAP_MIN,
)

if PUBLISH:
    targets.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(TARGET_TABLE)
    targets = spark.table(TARGET_TABLE)
else:
    print("PUBLISH=False: output tables were not written.")

display(targets.agg(
    F.count("*").alias("polls"),
    F.count("bike_target").alias("bike_labeled_polls"),
    F.avg("bike_target").alias("bike_onset_share"),
    F.count("dock_target").alias("dock_labeled_polls"),
    F.avg("dock_target").alias("dock_onset_share"),
))